In [2]:
import sys
sys.path.append('../')

from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE
from qiskit_ibm_runtime import Sampler
from qiskit.primitives import Sampler as LocalSampler
import matplotlib.pyplot as plt
import numpy as np

In [25]:
from qufold import MiyazawaJerniganInteraction

In [26]:
from qufold import Peptide, ProteinFoldingProblem, PenaltyParameters

In [34]:
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE

In [27]:
def build_pf(main_seq: str):
    """Builds the protein folding problem for the given sequence."""
    # Define the interaction
    side_chains = [""] * len(main_seq)

    mj_interaction = MiyazawaJerniganInteraction()

    penalty_back = 10
    penalty_chiral = 10
    penalty_1 = 10

    penalty_terms = PenaltyParameters(penalty_chiral, penalty_back, penalty_1)

    peptide = Peptide(main_seq, side_chains)

    protein_folding_problem = ProteinFoldingProblem(peptide, mj_interaction, penalty_terms)

    return protein_folding_problem

In [36]:

main_chain = "LHPGAGK" # chignolin
pf = build_pf(main_chain) #creates the PF problem instance

QiskitError: 'Pauli can only be multiplied by 1, -1j, -1, 1j.'

In [ ]:

qubit_op = pf.qubit_op() #creates the problem Hamiltonian

In [ ]:
from qiskit_ibm_runtime import Session, Options, QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum")
backend = service.backend("ibm_cleveland")

options = Options(
    execution={"shots": 5000},
    resilience_level=0,
    transpilation={"skip_transpilation": False},
    optimization_level=3,
    # environment={"job_tags": [f"batch_{i}"]},
)

#def run_vqe(qubit_op):
    # set classical optimizer
optimizer = COBYLA(maxiter=100)

    # set variational ansatz
ansatz = EfficientSU2(reps=1)

counts = []
values = []

def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)

In [ ]:
with Session(backend=backend):
    vqe = SamplingVQE(
            Sampler(options=options),
            ansatz=ansatz,
            optimizer=optimizer,
            aggregation=0.1,
            callback=store_intermediate_result)
    
    raw_result = vqe.compute_minimum_eigenvalue(qubit_op)

    #return raw_result, counts, values

In [ ]:
raw_result, counts, values = run_vqe(qubit_op)

In [ ]:
fig = plt.figure()

plt.plot(counts, values)
plt.ylabel("Conformation Energy")
plt.xlabel("VQE Iterations")

fig.add_axes([0.44, 0.51, 0.44, 0.32])

plt.plot(counts[40:], values[40:])
plt.ylabel("Conformation Energy")
plt.xlabel("VQE Iterations")
plt.show()

In [ ]:
result = pf.interpret(raw_result=raw_result)
print(
    "The bitstring representing the shape of the protein during optimization is: ",
    result.turn_sequence,
)
print("The expanded expression is:", result.get_result_binary_vector())

##

print(f"The folded protein's main sequence of turns is: {result.protein_shape_decoder.main_turns}")
print(f"and the side turn sequences are: {result.protein_shape_decoder.side_turns}")

fig = result.get_figure(title="3dcrd", ticks=False, grid=True)
fig.get_axes()[0].view_init(10, 70)

In [ ]:
# chignolin optimal solutions (2 degenerate)
opt_bitstring_1 = "0100001000011011011011"
opt_bitstring_2 = "0100001000011001011001"
opt_bitstring_3 = "0100001000001001001001"

result_1 = pf.interpret_bitstring(opt_bitstring_1)
result_2 = pf.interpret_bitstring(opt_bitstring_2)
result_3 = pf.interpret_bitstring(opt_bitstring_3)

In [ ]:
# multiply numerical values by 3.8
xyz_coords_1 = result_1.protein_shape_file_gen.get_xyz_data()
for i in range(len(xyz_coords_1)):
    for j in range(1, len(xyz_coords_1[i])):
        xyz_coords_1[i][j] = float(xyz_coords_1[i][j]) * 3.8

print(xyz_coords_1)

In [ ]:
xyz_coords_2 = result_2.protein_shape_file_gen.get_xyz_data()
for i in range(len(xyz_coords_2)):
    for j in range(1, len(xyz_coords_2[i])):
        xyz_coords_2[i][j] = float(xyz_coords_2[i][j]) * 3.8

print(xyz_coords_2)

In [ ]:
xyz_coords_3 = result_3.protein_shape_file_gen.get_xyz_data()
for i in range(len(xyz_coords_3)):
    for j in range(1, len(xyz_coords_3[i])):
        xyz_coords_3[i][j] = float(xyz_coords_3[i][j]) * 3.8

print(xyz_coords_3)